## How to run in Colab
1. Open this notebook on GitHub and click **Open in Colab** (or go to colab.research.google.com → File → Open notebook → GitHub and paste `simonlivson/crypto-market-explorer`).
2. Click **Runtime → Run all**.
3. I don't use any API key. If the CoinGecko or Coin Metrics API is down or rate-limited, my code automatically loads the saved CSV files from my GitHub repo instead, so every cell still runs.

# Crypto Market Explorer: Cycles, Altseasons and Coins That Break Away From Bitcoin

Most of the crypto market moves together with Bitcoin. But in some phases altcoins outperform (an "altseason"), and some coins break away from Bitcoin for months, like Hyperliquid and Zcash in 2025.

In this project I want to find out in which market phases diversification in crypto is real, and which coins are just "Bitcoin in disguise". My story has two parts:

- **Part A, the big picture:** Bitcoin and Ethereum daily history since 2013/2015, to study the 4-year cycle and altseasons.
- **Part B, inside the current phase:** the last 365 days for 12 coins, to see who follows Bitcoin and who breaks away.

Before I can analyze anything, I need data. This first section collects it.

# 1. Data Collection

In this section I collect three datasets:

| Dataset | Source | What it gives me |
|---|---|---|
| `coins_365d.csv` | CoinGecko API (free, no key) | daily price, market cap and volume for 12 coins, last 365 days |
| `btc_eth_history.csv` | Coin Metrics Community API (free, no key) | daily BTC and ETH prices since 2013 / 2015 |
| `coin_categories.csv` | my own file | the theme of each coin (Layer 1, Privacy, AI, ...) |

I save the raw API answers as JSON files and the cleaned tables as CSV files in my `data/` folder. If an API fails, I fall back to the CSV copies on GitHub.

## 1.1 Importing my libraries

I only need a few libraries here:
- `requests` sends HTTP requests to the APIs (like a browser does when it opens a page).
- `pandas` turns the answers into tables (DataFrames).
- `json` and `os` let me save the raw files to my folders.
- `time` lets me pause between requests so the free API doesn't block me.

I also find my `data/` folder. In Colab or from the repo root it is `data/`, but if I open the notebook from inside `notebooks/` (like in VS Code) it is `../data/`, so I check which one exists.

In [ ]:
import os
import json
import time

import requests
import pandas as pd

# my data folder: "data/" from the repo root or Colab, "../data/" from notebooks/
if os.path.isdir("data") or not os.path.isdir("../data"):
    DATA = "data/"
else:
    DATA = "../data/"
os.makedirs(DATA + "raw", exist_ok=True)
print("my data folder:", os.path.abspath(DATA))

# backup copies of my data on GitHub, in case an API fails
GITHUB_DATA = "https://raw.githubusercontent.com/simonlivson/crypto-market-explorer/main/data/"


## 1.2 Choosing my 12 coins

CoinGecko doesn't use tickers like "BTC". Every coin has its own **id** (for example `binancecoin` for BNB and `ripple` for XRP). I checked each id on CoinGecko, and I list them here with the ticker I'll use in my charts.

I picked large Layer 1 coins (BTC, ETH, SOL, ADA), BNB, XRP for payments, DOGE as a meme coin, LINK as an oracle, and four "theme" coins that had their own runs: HYPE (DEX/perps), ZEC and XMR (privacy) and TAO (AI).

In [ ]:
# CoinGecko id -> ticker
coins = {
    "bitcoin": "BTC",
    "ethereum": "ETH",
    "solana": "SOL",
    "binancecoin": "BNB",
    "ripple": "XRP",
    "dogecoin": "DOGE",
    "cardano": "ADA",
    "chainlink": "LINK",
    "hyperliquid": "HYPE",
    "zcash": "ZEC",
    "monero": "XMR",
    "bittensor": "TAO",
}
print(len(coins), "coins")


## 1.3 A safe way to call the API

The free CoinGecko API allows only a limited number of requests per minute. If I ask too fast, it answers with status code **429 (Too Many Requests)**.

So I write a small helper function `get_json()`:
- it tries the request up to 4 times,
- if the answer is 429 or a server error, it waits a bit longer each time (10, 20, 30 seconds) and tries again,
- if everything fails, it returns `None`, so my notebook doesn't crash.

In [ ]:
def get_json(url, params=None, tries=4):
    for attempt in range(1, tries + 1):
        try:
            response = requests.get(url, params=params, timeout=30)
            # 200 means everything is fine
            if response.status_code == 200:
                return response.json()
            print(f"  status {response.status_code}, attempt {attempt}")
        except requests.RequestException as error:
            print(f"  error: {error}, attempt {attempt}")
        # I wait longer after every failed try
        time.sleep(10 * attempt)
    return None


## 1.4 Fetching 365 days of data for each coin

For each coin I call the `market_chart` endpoint:

`https://api.coingecko.com/api/v3/coins/{id}/market_chart?vs_currency=usd&days=365&interval=daily`

The answer is JSON with three lists: `prices`, `market_caps` and `total_volumes`. Each item is a pair `[timestamp, value]`, where the timestamp is in milliseconds since 1970.

I save every answer as it is in `data/raw/<coin>.json`, so I always keep the original data. Between coins I pause 3 seconds to stay polite to the free API.

In [ ]:
url = "https://api.coingecko.com/api/v3/coins/{}/market_chart"
params = {"vs_currency": "usd", "days": 365, "interval": "daily"}

raw_data = {}
for coin_id in coins:
    print("fetching", coin_id)
    data = get_json(url.format(coin_id), params)
    if data is None:
        print("  failed:", coin_id)
        continue
    raw_data[coin_id] = data
    # I keep the original answer as a JSON file
    with open(f"{DATA}raw/{coin_id}.json", "w") as f:
        json.dump(data, f)
    time.sleep(3)

print(f"I got data for {len(raw_data)} of {len(coins)} coins")


## 1.5 Turning the JSON into one table

Now I turn the three lists of every coin into one **long-format** table. Long format means one row per coin per day, with these columns:

`date, coin, price, market_cap, volume`

This format is easy to filter (`df[df.coin == "BTC"]`), group (`groupby("coin")`) and pivot later.

Two small details:
- I convert the millisecond timestamps to dates with `pd.to_datetime(..., unit="ms")`.
- CoinGecko adds the current, unfinished day as the last point. I keep only one row per day and drop that last partial day, so every day is a complete one.

In [ ]:
tables = []
for coin_id, data in raw_data.items():
    # one small table per coin, one column per list
    prices = pd.DataFrame(data["prices"], columns=["ts", "price"])
    caps = pd.DataFrame(data["market_caps"], columns=["ts", "market_cap"])
    volumes = pd.DataFrame(data["total_volumes"], columns=["ts", "volume"])

    coin_df = prices.merge(caps, on="ts").merge(volumes, on="ts")
    coin_df["date"] = pd.to_datetime(coin_df["ts"], unit="ms").dt.normalize()
    coin_df["coin"] = coins[coin_id]
    tables.append(coin_df)

if tables:
    coins_df = pd.concat(tables, ignore_index=True)
    # one row per coin and day
    coins_df = coins_df.drop_duplicates(subset=["coin", "date"], keep="first")
    # the last day is still running, so I drop it
    coins_df = coins_df[coins_df["date"] < pd.Timestamp.today().normalize()]
    coins_df = coins_df[["date", "coin", "price", "market_cap", "volume"]]
else:
    coins_df = pd.DataFrame()

# if the API gave me nothing (or only part), I load my saved copy from GitHub
if len(raw_data) < len(coins):
    print("API failed, loading the saved CSV from GitHub")
    coins_df = pd.read_csv(GITHUB_DATA + "coins_365d.csv", parse_dates=["date"])
else:
    coins_df.to_csv(DATA + "coins_365d.csv", index=False)

coins_df.head()


### A first check of my coin data

I check how many rows each coin has and which dates they cover. A coin that is younger than one year (or listed later on CoinGecko) will have fewer rows. That matters later, because I can only compare coins over the days they all have.

In [ ]:
summary = coins_df.groupby("coin")["date"].agg(["count", "min", "max"])
summary.columns = ["days", "first_date", "last_date"]
print("total rows:", len(coins_df))
summary


## 1.6 Long-term Bitcoin and Ethereum history

CoinGecko's free tier only gives me 365 days of daily data, and one year is not enough to see a 4-year cycle. So for Part A I use the **Coin Metrics Community API**, which is free and needs no key:

`https://community-api.coinmetrics.io/v4/timeseries/asset-metrics?assets=btc,eth&metrics=PriceUSD&frequency=1d`

It returns a daily USD reference price for BTC (from 2010) and ETH (from August 2015). I start in 2013, as planned.

The answer comes in **pages**. Each page has a `next_page_url` if there's more data, so I keep following that link until there are no pages left.

In [ ]:
cm_url = "https://community-api.coinmetrics.io/v4/timeseries/asset-metrics"
cm_params = {
    "assets": "btc,eth",
    "metrics": "PriceUSD",
    "frequency": "1d",
    "start_time": "2013-01-01",
    "page_size": 10000,
}

rows = []
page = get_json(cm_url, cm_params)
while page is not None:
    rows.extend(page["data"])
    # I follow the next page until there is none
    next_url = page.get("next_page_url")
    if not next_url:
        break
    page = get_json(next_url)

if rows:
    with open(DATA + "raw/coinmetrics_btc_eth.json", "w") as f:
        json.dump(rows, f)
    history = pd.DataFrame(rows)
    history["date"] = pd.to_datetime(history["time"]).dt.tz_localize(None).dt.normalize()
    history["price"] = history["PriceUSD"].astype(float)
    # one column per coin: date, BTC, ETH
    history = history.pivot_table(index="date", columns="asset", values="price").reset_index()
    history.columns.name = None
    history = history.rename(columns={"btc": "BTC", "eth": "ETH"})
    history.to_csv(DATA + "btc_eth_history.csv", index=False)
else:
    print("API failed, loading the saved CSV from GitHub")
    history = pd.read_csv(GITHUB_DATA + "btc_eth_history.csv", parse_dates=["date"])

print(len(history), "days from", history["date"].min().date(), "to", history["date"].max().date())
print("first ETH price on", history.loc[history["ETH"].notna(), "date"].min().date())
history.tail()


Ethereum didn't exist before 2015, so its column is empty (NaN) for the early years. That's expected, not an error. I'll handle it in the cleaning section.

## 1.7 My coin categories

The last dataset is a small CSV I wrote by hand. It gives each coin a **theme**, so later I can check whether coins that break away from Bitcoin share a theme (hypothesis 8). I'll merge it with my price data using the `symbol` column.

In [ ]:
try:
    categories = pd.read_csv(DATA + "coin_categories.csv")
except FileNotFoundError:
    # in Colab the file isn't local, so I read it from GitHub
    categories = pd.read_csv(GITHUB_DATA + "coin_categories.csv")
categories


## 1.8 Summary of my data collection

I now have three datasets:
1. **coins_365d.csv**: 12 coins × about 365 days of price, market cap and volume from CoinGecko (raw JSON kept in `data/raw/`).
2. **btc_eth_history.csv**: daily BTC prices since 2013 and ETH since 2015 from Coin Metrics.
3. **coin_categories.csv**: the theme of every coin.

Next, in section 2, I clean the data: missing values, duplicates, data types and outliers.

# 2. Data Cleaning

Raw data is almost never ready for analysis. Before I trust any chart or test, I check my two price tables step by step:

1. **Data types**: are dates real dates and numbers real numbers?
2. **Missing values**: where are there gaps, and why?
3. **Duplicates**: does any coin have the same day twice?
4. **Sorting**: are the rows in time order? (needed for returns)
5. **Sanity checks**: no zero or negative prices, the same number of days for every coin.
6. **Outliers**: which days had extreme moves, and are they errors or real?
7. **Merging and new columns**: I add the coin themes and daily returns.

At the end I save two clean files: `coins_clean.csv` and `btc_eth_clean.csv`.

To make this section work on its own, I first reload both tables from my CSV files (or from GitHub if the local files aren't there).

In [ ]:
import numpy as np

def load_csv(name):
    # I use my local copy first, and the GitHub copy as a backup
    if os.path.exists(DATA + name):
        return pd.read_csv(DATA + name)
    return pd.read_csv(GITHUB_DATA + name)

coins_df = load_csv("coins_365d.csv")
history = load_csv("btc_eth_history.csv")
categories = load_csv("coin_categories.csv")
print(coins_df.shape, history.shape, categories.shape)


## 2.1 Data types

`info()` shows me every column, its type (`dtype`) and how many values are not empty.

When pandas reads a CSV, dates come in as plain text (`object`). Text dates can't be sorted by time or used for things like "day of the week", so I convert them with `pd.to_datetime()`. Prices, market caps and volumes should be decimal numbers (`float64`). I convert them with `pd.to_numeric(..., errors="coerce")`: if a value isn't a number, it becomes `NaN` instead of crashing, and I'll find it in the missing values step.

In [ ]:
coins_df.info()
print()
history.info()


In [ ]:
# text dates -> real dates
coins_df["date"] = pd.to_datetime(coins_df["date"])
history["date"] = pd.to_datetime(history["date"])

# all number columns as floats
for col in ["price", "market_cap", "volume"]:
    coins_df[col] = pd.to_numeric(coins_df[col], errors="coerce").astype(float)
for col in ["BTC", "ETH"]:
    history[col] = pd.to_numeric(history[col], errors="coerce").astype(float)

print(coins_df.dtypes, "\n")
print(history.dtypes)


## 2.2 Missing values

`isna().sum()` counts the empty cells in every column. For each gap I want to know *why* it's there before I decide what to do:

- In my **coin table**, a missing price or volume would mean CoinGecko had no data for that day.
- In my **history table**, I already know ETH is empty before **August 2015**, simply because Ethereum didn't exist yet. Those gaps are not errors. I keep those rows, because BTC has real prices there and I need them for the long cycle. When I analyze ETH, I'll just use the rows where ETH has a value.

In [ ]:
print("missing values in my coin table:")
print(coins_df.isna().sum(), "\n")
print("missing values in my history table:")
print(history.isna().sum())


In [ ]:
# where the ETH gap is: it should end right before Ethereum started trading
eth_missing = history[history["ETH"].isna()]
print("ETH missing from", eth_missing["date"].min().date(), "to", eth_missing["date"].max().date())
print("first ETH price:", history.loc[history["ETH"].notna(), "date"].min().date())

# any other gap in ETH after it started would be a real problem
first_eth = history.loc[history["ETH"].notna(), "date"].min()
print("ETH gaps after launch:", history.loc[history["date"] >= first_eth, "ETH"].isna().sum())


In [ ]:
# rows in my coin table with a missing number, if there are any
missing_rows = coins_df[coins_df[["price", "market_cap", "volume"]].isna().any(axis=1)]
print(len(missing_rows), "rows with a missing value")
missing_rows


**My decision:** I keep the ETH gap in the history table (it's real history, not an error). If my coin table has any rows with a missing price, I drop them, because a row without a price can't be used for returns. A missing market cap or volume alone I fill from the day before with `ffill()` inside each coin, since these numbers change slowly from one day to the next.

In [ ]:
# a row without a price is useless for me
coins_df = coins_df.dropna(subset=["price"])

# small gaps in market cap / volume: I take the value of the day before, per coin
coins_df = coins_df.sort_values(["coin", "date"])
coins_df[["market_cap", "volume"]] = coins_df.groupby("coin")[["market_cap", "volume"]].ffill()
print(coins_df.isna().sum())


## 2.3 Duplicates

A duplicate here means the same coin appears twice on the same day. That could happen if an API sends two points for one day. I check with `duplicated()` on the columns `date` + `coin` (and only `date` in the history table), and I keep the first row if I find any.

In [ ]:
dup_coins = coins_df.duplicated(subset=["date", "coin"]).sum()
dup_hist = history.duplicated(subset=["date"]).sum()
print("duplicates in my coin table:", dup_coins)
print("duplicates in my history table:", dup_hist)

coins_df = coins_df.drop_duplicates(subset=["date", "coin"], keep="first")
history = history.drop_duplicates(subset=["date"], keep="first")


## 2.4 Sorting

Daily returns compare each day with the day before, so the rows must be in time order. I sort my coin table by coin and then date, and my history table by date. `reset_index(drop=True)` gives me a clean 0, 1, 2, ... index again.

In [ ]:
coins_df = coins_df.sort_values(["coin", "date"]).reset_index(drop=True)
history = history.sort_values("date").reset_index(drop=True)
coins_df.head()


## 2.5 Sanity checks

Now I check that the numbers make sense:
- A price, market cap or volume can never be zero or negative.
- Every coin should have the same number of days (364), with no missing days in between.
- My history table should have one row for every calendar day, because crypto trades 7 days a week.

In [ ]:
# zero or negative numbers
for col in ["price", "market_cap", "volume"]:
    print(col, "<= 0:", (coins_df[col] <= 0).sum())
print("BTC <= 0:", (history["BTC"] <= 0).sum())
print("ETH <= 0:", (history["ETH"] <= 0).sum())


In [ ]:
# number of days and date range per coin
days = coins_df.groupby("coin")["date"].agg(["count", "min", "max"])
# days between first and last date + 1 = days there should be
days["expected"] = (days["max"] - days["min"]).dt.days + 1
days["missing_days"] = days["expected"] - days["count"]
days


In [ ]:
# the history table should have every calendar day
all_days = pd.date_range(history["date"].min(), history["date"].max(), freq="D")
print("days in my history:", len(history), "| calendar days:", len(all_days))
print("missing calendar days:", len(all_days.difference(history["date"])))


## 2.6 Merging the coin themes

My `coin_categories.csv` has the theme of each coin. I merge it onto my coin table with `merge()`, matching my `coin` column (the ticker, like "BTC") with the `symbol` column. I use `how="left"` so I keep all my price rows, and then I check that no coin ended up without a theme.

In [ ]:
categories = categories.rename(columns={"coin": "coingecko_id", "symbol": "coin"})
coins_df = coins_df.merge(categories, on="coin", how="left")

print("rows without a theme:", coins_df["theme"].isna().sum())
coins_df.groupby("theme")["coin"].unique()


## 2.7 New columns: daily returns

Prices are hard to compare between coins: BTC costs about \$100,000 and DOGE about \$0.20. So for most of my analysis I use **returns**, the change in percent from one day to the next:

- **daily_return** = today's price / yesterday's price - 1, made with `pct_change()`. A value of 0.05 means +5%.
- **log_return** = ln(today / yesterday). It's almost the same as the daily return for small moves, but log returns can simply be added up over time, which is why they're popular in finance.

I calculate them **per coin** with `groupby("coin")`, otherwise the first day of ETH would be compared with the last day of BTC. The first day of each coin has no "yesterday", so its return is `NaN`, which is expected.

In [ ]:
coins_df["daily_return"] = coins_df.groupby("coin")["price"].pct_change()
coins_df["log_return"] = np.log(coins_df["price"] / coins_df.groupby("coin")["price"].shift(1))

# the same for my long BTC and ETH history
for col in ["BTC", "ETH"]:
    history[col + "_return"] = history[col].pct_change(fill_method=None)
    history[col + "_log_return"] = np.log(history[col] / history[col].shift(1))

coins_df[["date", "coin", "price", "daily_return", "log_return"]].head()


I also add a few date columns that I'll use later for seasonality and cycles: the weekday, the month and the year.

In [ ]:
coins_df["weekday"] = coins_df["date"].dt.day_name()
coins_df["month"] = coins_df["date"].dt.month
history["weekday"] = history["date"].dt.day_name()
history["month"] = history["date"].dt.month
history["year"] = history["date"].dt.year


## 2.8 Outliers

An outlier is a value far away from the usual ones. I look for them in the **daily returns**, not in the prices, because a high price isn't strange, but a +40% day is.

I use the **IQR method** that I learned in the course:
- Q1 is the 25% value and Q3 the 75% value of a coin's returns, and IQR = Q3 - Q1.
- A return is an outlier if it's below Q1 - 1.5 × IQR or above Q3 + 1.5 × IQR.

I calculate the limits for **each coin separately**, because a normal day for a small coin like ZEC would be an extreme day for BTC.

**Important:** in crypto, these outliers are usually *real* moves (crashes, hype rallies, news), not data errors. They're exactly what my breakaway story is about. So I **don't delete them**. I only flag them in a new column `is_outlier`, so I can study them or leave them out on purpose later.

One side effect of per-coin limits: a very volatile coin like ZEC or HYPE gets much wider limits, so it can show *fewer* outlier days than a calmer coin like ETH, even though its biggest moves are larger.

In [ ]:
def iqr_flag(returns):
    q1 = returns.quantile(0.25)
    q3 = returns.quantile(0.75)
    iqr = q3 - q1
    # True when the return is outside the IQR limits
    return (returns < q1 - 1.5 * iqr) | (returns > q3 + 1.5 * iqr)

coins_df["is_outlier"] = coins_df.groupby("coin")["daily_return"].transform(iqr_flag)

outliers = coins_df.groupby("coin")["is_outlier"].sum().sort_values(ascending=False)
print("outlier days per coin:")
outliers


In [ ]:
# my 10 most extreme days, up or down
coins_df["abs_return"] = coins_df["daily_return"].abs()
top_moves = coins_df.sort_values("abs_return", ascending=False).head(10)
coins_df = coins_df.drop(columns="abs_return")
top_moves[["date", "coin", "theme", "price", "daily_return", "volume"]]


As a second check, I also look at the **z-score**: how many standard deviations a return is away from the coin's average. Above 3 is very rare in a normal distribution. If crypto returns were "normal", only about 0.3% of days would pass that limit.

In [ ]:
def zscore(returns):
    return (returns - returns.mean()) / returns.std()

coins_df["z_score"] = coins_df.groupby("coin")["daily_return"].transform(zscore)
z_count = (coins_df["z_score"].abs() > 3).groupby(coins_df["coin"]).sum()
print("days with |z| > 3 per coin:")
z_count.sort_values(ascending=False)


In [ ]:
# the same IQR flag for my long BTC and ETH history
for col in ["BTC", "ETH"]:
    history[col + "_outlier"] = iqr_flag(history[col + "_return"])
    print(col, "outlier days:", history[col + "_outlier"].sum())

# my biggest BTC days since 2013
history.loc[history["BTC_return"].abs().sort_values(ascending=False).index[:5], ["date", "BTC", "BTC_return"]]


## 2.9 Saving my clean data

My data is now clean: correct types, no duplicates, sorted, checked, with themes, returns and outlier flags. I save both tables so the next sections (and my Streamlit dashboard) can start from the clean versions.

In [ ]:
coins_df.to_csv(DATA + "coins_clean.csv", index=False)
history.to_csv(DATA + "btc_eth_clean.csv", index=False)
print("coins_clean:", coins_df.shape)
print("btc_eth_clean:", history.shape)


## 2.10 Cleaning summary

- **Types:** I turned dates into `datetime` and all numbers into `float`.
- **Missing values:** the only gap is ETH before August 2015, which is real (Ethereum didn't exist yet), so I kept it.
- **Duplicates and sorting:** I removed any repeated coin-days and sorted everything by time.
- **Sanity checks:** no zero or negative values, and every coin covers the same days.
- **New columns:** theme, daily and log returns, weekday, month and year.
- **Outliers:** crypto has many extreme days. They're real market moves, so I kept them and flagged them with `is_outlier` and `z_score`.

Next, in section 3, I explore the data.

# 3. Part A: The Big Picture — Bitcoin Cycles and Altseasons

Before I look at single coins, I want to understand the market "weather". Many crypto investors believe Bitcoin moves in a **4-year cycle**: roughly 3 years of rising prices (a **bull** market) and 1 year of falling prices (a **bear** market). The cycle is often linked to the Bitcoin **halving**, the day every ~4 years when the reward for mining new Bitcoin is cut in half, so fewer new coins come to the market.

In this part I test three hypotheses with my long BTC and ETH history:
1. Bitcoin follows a roughly 4-year cycle: about 3 bull years and 1 bear year.
2. Bull and bear phases have significantly different returns and volatility.
3. Altseasons (ETH rising strongly against BTC) happen late in bull phases.

I start from my clean files from section 2.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy import stats

history = load_csv("btc_eth_clean.csv")
history["date"] = pd.to_datetime(history["date"])
coins_df = load_csv("coins_clean.csv")
coins_df["date"] = pd.to_datetime(coins_df["date"])

sns.set_style("whitegrid")
print(history["date"].min().date(), "to", history["date"].max().date())


## 3.1 Bitcoin's long-term price and the halvings

Bitcoin went from about \$13 in January 2013 to over \$100,000. On a normal chart, everything before 2017 would look like a flat line. So I use a **log scale** on the y-axis: every step up means the price is multiplied by 10 (\$10 → \$100 → \$1,000 ...). On a log chart, the same *percent* move always looks the same size, which is fair when I compare cycles.

I mark the four Bitcoin halvings as dashed lines. The dates come from the Bitcoin blockchain (block 210,000, 420,000, 630,000 and 840,000):
- 2012-11-28 (before my data starts, so it isn't on the chart)
- 2016-07-09
- 2020-05-11
- 2024-04-20

In [ ]:
halvings = pd.to_datetime(["2012-11-28", "2016-07-09", "2020-05-11", "2024-04-20"])

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(history["date"], history["BTC"], color="orange", linewidth=1.2, label="BTC price")
ax.set_yscale("log")

# a dashed line for every halving inside my data
for h in halvings:
    if h >= history["date"].min():
        ax.axvline(h, color="black", linestyle="--", linewidth=1)
        ax.text(h, ax.get_ylim()[1] * 0.6, " halving\n " + str(h.date()), fontsize=8)

ax.set_title("Bitcoin price since 2013 (log scale) with halvings")
ax.set_xlabel("Date")
ax.set_ylabel("Price in USD (log scale)")
ax.legend()
plt.show()


## 3.2 Finding bull and bear phases from the data

I don't want to choose the phases by eye, so I use a clear rule based on **drawdown**:

- The **running peak** is the highest price so far (`cummax()`).
- The **drawdown** is how far the price is below that peak: `price / running peak - 1`. A drawdown of -0.80 means the price is 80% below its all-time high.

Every time Bitcoin makes a new all-time high, a new "episode" starts. Inside each episode I find the lowest point.

**My rule:** a **bear phase** goes from an all-time high to the lowest point after it, if the price fell **at least 50%** and the fall lasted **at least 180 days**. The 180 days filter out short crashes (like April 2013 or the summer of 2021) that recovered within a few months. Everything between the bottom of one bear and the peak of the next is a **bull phase**.

The last phase is special: the newest bear started at the October 2025 peak, and its bottom is only "the lowest point so far". I keep it in my table but label it as ongoing, and I don't call the time after that bottom a bull yet, because I can't know if the bottom is final.

In [ ]:
btc = history[["date", "BTC", "BTC_return"]].copy()
btc["running_peak"] = btc["BTC"].cummax()
btc["drawdown"] = btc["BTC"] / btc["running_peak"] - 1

# a new episode starts at every new all-time high
btc["episode"] = (btc["BTC"] >= btc["running_peak"]).cumsum()

bears = []
for ep, part in btc.groupby("episode"):
    bottom = part.loc[part["drawdown"].idxmin()]
    days = (bottom["date"] - part["date"].iloc[0]).days
    # my rule: at least -50% and at least 180 days
    if bottom["drawdown"] <= -0.5 and days >= 180:
        bears.append((part["date"].iloc[0], bottom["date"], bottom["drawdown"]))

for peak, bottom, dd in bears:
    print("bear:", peak.date(), "->", bottom.date(), f"drawdown {dd:.0%}")


Now I build the full list of phases: the data start to the first peak is a bull, then each bear, then the bull from its bottom to the next peak, and so on.

In [ ]:
phases = []
start = btc["date"].iloc[0]
for peak, bottom, dd in bears:
    phases.append(("Bull", start, peak))
    phases.append(("Bear", peak, bottom))
    start = bottom
# the time after the last bottom: too early to call it
phases.append(("After last low (ongoing)", start, btc["date"].iloc[-1]))

def phase_stats(name, start, end):
    part = btc[(btc["date"] > start) & (btc["date"] <= end)]
    first_price = btc.loc[btc["date"] == start, "BTC"].iloc[0]
    last_price = btc.loc[btc["date"] == end, "BTC"].iloc[0]
    return {
        "phase": name,
        "start": start.date(),
        "end": end.date(),
        "days": (end - start).days,
        "years": round((end - start).days / 365.25, 2),
        "total_return_%": round((last_price / first_price - 1) * 100, 1),
        "avg_daily_return_%": round(part["BTC_return"].mean() * 100, 3),
        "daily_volatility_%": round(part["BTC_return"].std() * 100, 2),
    }

phase_table = pd.DataFrame([phase_stats(*p) for p in phases])
# the first bull starts with my data, not at a real bottom, and the last bear may not be over
phase_table.loc[0, "phase"] = "Bull (partial, data starts 2013)"
phase_table.loc[len(phase_table) - 2, "phase"] = "Bear (ongoing, low so far)"
phase_table


I shade the phases on the price chart: green for bull, red for bear and grey for the open period after the latest low.

In [ ]:
colors = {"Bull": "green", "Bear": "red"}

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(btc["date"], btc["BTC"], color="black", linewidth=1)
ax.set_yscale("log")

for name, start, end in phases:
    ax.axvspan(start, end, color=colors.get(name, "grey"), alpha=0.15)
for h in halvings[1:]:
    ax.axvline(h, color="blue", linestyle="--", linewidth=1)

ax.set_title("Bitcoin bull (green) and bear (red) phases, halvings as blue dashed lines")
ax.set_xlabel("Date")
ax.set_ylabel("Price in USD (log scale)")
plt.show()


## 3.3 Testing hypotheses 1 and 2

### Hypothesis 1: about 3 bull years and 1 bear year

I compare the length of the **complete** phases only. The first bull is cut off by the start of my data, and the latest bear and the time after it aren't finished, so they would distort the averages.

In [ ]:
complete = phase_table[phase_table["phase"].isin(["Bull", "Bear"])]
print(complete[["phase", "start", "end", "years"]], "\n")
print("average length in years:")
complete.groupby("phase")["years"].mean().round(2)


### Hypothesis 2: bull and bear days really are different

Bull markets obviously go up and bear markets go down. The question is whether the **daily** returns are different enough that it isn't just luck. For this I use a **t-test**, which compares the averages of two groups.

- I use **Welch's t-test** (`ttest_ind` with `equal_var=False`), because bull and bear days don't have the same spread.
- The **null hypothesis** is "the average daily return is the same in bull and bear phases".
- The **p-value** is the chance of seeing a difference this big if the null hypothesis were true. If it's below 0.05, I reject the null and say the difference is real (statistically significant).

I also compare the **volatility** (standard deviation of daily returns) and check it with **Levene's test**, which asks whether two groups have the same spread.

I use all days that belong to a bull or bear phase, including the partial and ongoing ones, because each single day is still clearly in a rising or falling market.

In [ ]:
# every day gets the name of its phase
btc["phase"] = None
for name, start, end in phases:
    btc.loc[(btc["date"] > start) & (btc["date"] <= end), "phase"] = name

bull_days = btc.loc[btc["phase"] == "Bull", "BTC_return"].dropna()
bear_days = btc.loc[btc["phase"] == "Bear", "BTC_return"].dropna()

t_stat, p_value = stats.ttest_ind(bull_days, bear_days, equal_var=False)
print(f"bull days: {len(bull_days)}, average {bull_days.mean():.3%}, volatility {bull_days.std():.2%}")
print(f"bear days: {len(bear_days)}, average {bear_days.mean():.3%}, volatility {bear_days.std():.2%}")
print(f"Welch t-test: t = {t_stat:.2f}, p = {p_value:.5f}")

lev_stat, lev_p = stats.levene(bull_days, bear_days)
print(f"Levene test (volatility): p = {lev_p:.5f}")


In [ ]:
# the two distributions next to each other
fig, ax = plt.subplots(figsize=(10, 4))
sns.histplot(bull_days * 100, bins=100, color="green", alpha=0.5, label="Bull days", stat="density", ax=ax)
sns.histplot(bear_days * 100, bins=100, color="red", alpha=0.5, label="Bear days", stat="density", ax=ax)
ax.set_xlim(-20, 20)
ax.set_title("Daily BTC returns in bull vs bear phases")
ax.set_xlabel("Daily return (%)")
ax.set_ylabel("Density")
ax.legend()
plt.show()


## 3.4 Bitcoin returns by year and by month

Now I look at the calendar. For each year I calculate the total return from the first to the last price of the year with `groupby("year")`. Then I build a **pivot table** with one row per year and one column per month, and show it as a heatmap. Green cells are good months, red cells bad months.

For the monthly table I first take the last price of every month with `resample("ME")`, then use `pct_change()` to get the monthly return. The current year (2026) is still incomplete.

In [ ]:
yearly = btc.groupby(btc["date"].dt.year)["BTC"].agg(["first", "last"])
yearly["return_%"] = ((yearly["last"] / yearly["first"] - 1) * 100).round(1)
yearly.index.name = "year"
print("best year:", yearly["return_%"].idxmax(), yearly["return_%"].max(), "%")
print("worst year:", yearly["return_%"].idxmin(), yearly["return_%"].min(), "%")
yearly


In [ ]:
# last price of every month, then the change from month to month
monthly = btc.set_index("date")["BTC"].resample("ME").last().pct_change().to_frame("return")
monthly["year"] = monthly.index.year
monthly["month"] = monthly.index.month

month_table = monthly.pivot_table(index="year", columns="month", values="return") * 100

fig, ax = plt.subplots(figsize=(13, 7))
sns.heatmap(month_table, annot=True, fmt=".0f", cmap="RdYlGn", center=0, cbar_kws={"label": "Monthly return (%)"}, ax=ax)
ax.set_title("Bitcoin monthly returns (%) by year")
ax.set_xlabel("Month")
ax.set_ylabel("Year")
plt.show()


In [ ]:
# average and median return of each month over all years
month_avg = month_table.agg(["mean", "median"]).T.round(1)
month_avg["positive_years"] = (month_table > 0).sum()
month_avg


## 3.5 Altseasons: when does Ethereum beat Bitcoin?

An **altseason** is a period when altcoins rise much faster than Bitcoin. My best long-term signal is the **ETH/BTC ratio**: the price of one ETH measured in BTC. When the ratio goes up, ETH is beating BTC, and since ETH is the biggest altcoin, money usually flows to the rest of the altcoins after it.

**My rule for an altseason period:** the ETH/BTC ratio rose by **at least 50% over 90 days**. I mark every day where that's true, join days that are close together (less than 30 days apart) into one period, and then check in which Bitcoin phase each period falls (hypothesis 3).

In [ ]:
eth = history[["date", "BTC", "ETH"]].dropna().copy()
eth["eth_btc"] = eth["ETH"] / eth["BTC"]
# how much the ratio changed over the last 90 days
eth["change_90d"] = eth["eth_btc"].pct_change(90)
eth["alt_signal"] = eth["change_90d"] >= 0.5

signal_days = eth.loc[eth["alt_signal"], "date"]
# a gap of more than 30 days starts a new period
period_id = (signal_days.diff().dt.days > 30).cumsum()
alt_periods = signal_days.groupby(period_id).agg(["min", "max"])
alt_periods.columns = ["start", "end"]
alt_periods["days"] = (alt_periods["end"] - alt_periods["start"]).dt.days + 1
alt_periods


To check hypothesis 3, I look up the Bitcoin phase at the middle of each period, and where it sits inside that phase: 0% means the phase just started, 100% means it's about to end. "Late in a bull phase" would mean a high percentage in a Bull phase.

In [ ]:
def where_in_cycle(day):
    for name, start, end in phases:
        if start < day <= end:
            share = (day - start).days / (end - start).days
            return name, round(share * 100)
    return None, None

middles = alt_periods["start"] + (alt_periods["end"] - alt_periods["start"]) / 2
alt_periods["btc_phase"] = [where_in_cycle(d)[0] for d in middles]
alt_periods["position_in_phase_%"] = [where_in_cycle(d)[1] for d in middles]
alt_periods


In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(eth["date"], eth["eth_btc"], color="purple", linewidth=1, label="ETH/BTC")

for name, start, end in phases:
    ax.axvspan(start, end, color=colors.get(name, "grey"), alpha=0.1)
# my altseason periods in yellow
for _, row in alt_periods.iterrows():
    ax.axvspan(row["start"], row["end"], color="gold", alpha=0.6)

ax.set_title("ETH/BTC ratio with altseason periods (yellow) and BTC bull/bear phases")
ax.set_xlabel("Date")
ax.set_ylabel("1 ETH in BTC")
ax.legend()
plt.show()


### What about 2025 and 2026?

The ETH/BTC ratio only covers one altcoin. For the last year I can do better with my 12 coins: every day I check which of my **11 altcoins** beat Bitcoin over the last 90 days. That's a small version of the well-known "Altcoin Season Index", which calls it an altseason when **75%** of the top coins beat Bitcoin over 90 days.

In [ ]:
prices = coins_df.pivot_table(index="date", columns="coin", values="price")
change_90d = prices.pct_change(90).dropna()

alts = [c for c in change_90d.columns if c != "BTC"]
# True when an altcoin did better than BTC over 90 days
beat_btc = change_90d[alts].gt(change_90d["BTC"], axis=0)
alt_score = beat_btc.mean(axis=1) * 100

print("days checked:", len(alt_score), alt_score.index.min().date(), "to", alt_score.index.max().date())
print("highest share of altcoins beating BTC:", round(alt_score.max()), "% on", alt_score.idxmax().date())
print("days with 75% or more:", (alt_score >= 75).sum())
print("average share:", round(alt_score.mean()), "%")


In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(alt_score.index, alt_score, color="teal")
ax.axhline(75, color="red", linestyle="--", label="altseason line (75%)")
ax.set_ylim(0, 100)
ax.set_title("My altseason score: share of 11 altcoins beating BTC over 90 days")
ax.set_xlabel("Date")
ax.set_ylabel("Altcoins beating BTC (%)")
ax.legend()
plt.show()

# which coins beat BTC most often
(beat_btc.mean() * 100).round().sort_values(ascending=False)


## 3.6 What I learned in Part A

I write my conclusions in the summary below. The numbers come from the tables above.

**Hypothesis 1 (≈3 bull years, ≈1 bear year): supported.** My drawdown rule found three complete cycles since 2014, and they're surprisingly regular. The bulls lasted 2.92, 2.90 and 2.91 years, and the bears 1.11, 1.00 and 1.00 years. On average that's **2.91 bull years and 1.04 bear years**. The bear markets ended at -85%, -84% and -77%. The newest bear started at the all-time high on 2025-10-06, almost exactly 4 years after the 2021 peak. Its lowest point so far was -53% on 2026-06-30. That's much shallower than earlier bears, and I can't say yet whether it's over. With only three complete cycles, this is a pattern, not proof.

**Hypothesis 2 (bull and bear phases differ): supported for returns, only weakly for volatility.** The average daily return was +0.49% on bull days and -0.34% on bear days. Welch's t-test gives p < 0.00001, so the difference is very unlikely to be luck. Volatility was almost the same (3.94% vs 4.02% per day). Levene's p = 0.048 is just under 0.05, so the difference is barely significant. In practice, Bitcoin is about equally wild in both phases. Only the direction changes. Each cycle is also calmer than the one before: daily volatility fell from about 7% in 2013 to about 2.4% in 2025–2026.

**Calendar:** the best year was 2013 (+5,373%) and the worst was 2018 (-72.6%). Every bear year (2014, 2018, 2022) lost between 57% and 73%. October was positive in 10 of 13 years. Some monthly averages are pulled up by single extreme months, like November 2013, so I trust the median more than the mean.

**Hypothesis 3 (altseasons come late in bull phases): partly supported.** I found 8 periods where ETH/BTC rose 50% or more over 90 days. 6 fell inside bull phases. The longest and strongest were in 2017 (81% into the bull), May–July 2021 (86%) and July–October 2025 (96%, right before the BTC peak). The 2016 run was in the middle of a bull (41%), and two short ones happened inside bears.

**2025–2026:** ETH had a real run against BTC in summer 2025, but my 12-coin score only starts in February 2026 (I need 90 days of history first). From February to August 2026, the share of altcoins beating BTC stayed far below 75%. On average only 45% of my altcoins beat BTC. The leaders were the theme coins HYPE (93% of days), XMR (82%) and ZEC (75%), while XRP and DOGE almost never beat BTC. So for most of the year, money went into a few narratives, not the whole altcoin market. A short altseason-like window came only after the June low: 15 days between 2026-09-09 and 2026-10-24, with a high of 91%, while BTC itself rose 23–42% over 90 days. Over the same months, ZEC rose 188%. That window is short and based on only 11 coins, so I treat it as a hint, not a full altseason.

# 4. Part B: Which coins break away from Bitcoin?

In Part A I saw that the market as a whole follows Bitcoin's cycles. Now I zoom in on my 12 coins over the last year and ask a more nuanced question: **which coins stop following Bitcoin, when, and why?**

My plan:
1. **Correlation** of daily returns: how similar is each coin's daily move to BTC's? (hypothesis 4)
2. **Beta**: when BTC moves 1%, how much does each coin move on average? (hypothesis 5)
3. **Rolling correlation** over 30 and 90 days, and my own rule for a **breakaway** period (hypothesis 6)
4. **Volume**: do breakaways come with unusual trading volume? (hypothesis 7)
5. **Themes**: do breakaways cluster in certain themes, like privacy or AI? (hypothesis 8)

In [ ]:
coins_df = load_csv("coins_clean.csv")
coins_df["date"] = pd.to_datetime(coins_df["date"])

# one column per coin, one row per day
returns = coins_df.pivot_table(index="date", columns="coin", values="daily_return").dropna()
volumes = coins_df.pivot_table(index="date", columns="coin", values="volume")
prices = coins_df.pivot_table(index="date", columns="coin", values="price")
themes = coins_df.groupby("coin")["theme"].first()

alts = [c for c in returns.columns if c != "BTC"]
print("days:", len(returns), "| coins:", list(returns.columns))

## 4.1 Hypothesis 4: altcoins are highly correlated with BTC

The **correlation** (Pearson's r) goes from -1 to +1. +1 means two coins always move in the same direction by the same relative amount, 0 means no link at all. I calculate it on **daily returns**, not prices, because two prices that both trend up look correlated even when their daily moves have nothing to do with each other.

`returns.corr()` gives me the full matrix: every coin against every other coin.

In [ ]:
corr = returns.corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, ax=ax)
ax.set_title("Correlation of daily returns (last year)")
plt.show()

# each altcoin vs BTC, highest first
corr_btc = corr["BTC"].drop("BTC").sort_values(ascending=False)
print("average correlation with BTC:", round(corr_btc.mean(), 2))
corr_btc.round(2)

I also want to know if the correlations are statistically real and not just luck. `stats.pearsonr()` gives me r and a p-value for each coin. A p-value below 0.05 means the link is very unlikely to be random.

In [ ]:
for coin in corr_btc.index:
    r, p = stats.pearsonr(returns["BTC"], returns[coin])
    print(f"{coin:5} r = {r:.2f}   p = {p:.1e}")

## 4.2 Hypothesis 5: beta to BTC

Correlation tells me *how often* a coin moves with BTC, but not *how much*. For that I use **beta**:

beta = covariance(coin, BTC) / variance(BTC)

- beta = 1: the coin moves as much as BTC
- beta = 2: when BTC falls 1%, the coin falls about 2% on average
- beta below 1: the coin moves less than BTC (or ignores it)

I also show the R² (correlation squared): the share of a coin's daily moves that BTC "explains".

In [ ]:
btc_var = returns["BTC"].var()
beta = {}
for coin in alts:
    beta[coin] = returns[coin].cov(returns["BTC"]) / btc_var

beta_table = pd.DataFrame({"beta": pd.Series(beta), "correlation": corr["BTC"][alts]})
beta_table["r_squared_%"] = beta_table["correlation"] ** 2 * 100
beta_table["theme"] = themes[alts]
beta_table = beta_table.sort_values("beta", ascending=False).round(2)
beta_table

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(beta_table.index, beta_table["beta"], color="steelblue")
ax.axhline(1, color="red", linestyle="--", label="beta = 1 (moves like BTC)")
ax.set_title("Beta to BTC: how much each coin moves when BTC moves 1%")
ax.set_ylabel("beta")
ax.legend()
plt.show()

## 4.3 Hypothesis 6: rolling correlation and breakaways

One correlation number for the whole year hides a lot. A coin can follow BTC for months and then suddenly go its own way. So I calculate a **rolling correlation**: for every day, the correlation over the last 30 (or 90) days only. `rolling(30).corr()` does this for me.

In [ ]:
roll_30 = returns[alts].rolling(30).corr(returns["BTC"])
roll_90 = returns[alts].rolling(90).corr(returns["BTC"])

fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)
for coin in alts:
    axes[0].plot(roll_30.index, roll_30[coin], linewidth=1, label=coin)
    axes[1].plot(roll_90.index, roll_90[coin], linewidth=1, label=coin)
axes[0].set_title("30-day rolling correlation with BTC")
axes[1].set_title("90-day rolling correlation with BTC")
axes[0].legend(ncol=6, fontsize=8)
plt.show()

print("lowest 30-day correlation per coin:")
roll_30.min().sort_values().round(2)

### My breakaway rule

A low correlation alone isn't interesting if the coin is just falling on its own. I call a day a **breakaway day** when both are true:
1. the coin's 30-day correlation with BTC is **below 0.4** (it stopped following BTC), and
2. the coin **beat BTC by more than 20%** over the same 30 days (it went its own way *upward*).

Then I join breakaway days that are close together (gaps of 7 days or less) into **breakaway periods**, with the same `cumsum()` trick I used in Part A.

In [ ]:
change_30 = prices.pct_change(30)
# how much better than BTC each coin did over 30 days
excess_30 = change_30[alts].sub(change_30["BTC"], axis=0)

breakaway = (roll_30 < 0.4) & (excess_30.loc[roll_30.index] > 0.2)
print("breakaway days per coin:")
breakaway.sum().sort_values(ascending=False)

In [ ]:
periods = []
for coin in alts:
    days = breakaway.index[breakaway[coin]]
    if len(days) == 0:
        continue
    # a gap of more than 7 days starts a new period
    period_id = (pd.Series(days).diff().dt.days > 7).cumsum()
    for _, group in pd.Series(days).groupby(period_id.values):
        start, end = group.min(), group.max()
        periods.append({
            "coin": coin,
            "theme": themes[coin],
            "start": start.date(),
            "end": end.date(),
            "days": len(group),
            "min_corr_30d": round(roll_30.loc[start:end, coin].min(), 2),
            "max_beat_btc_%": round(excess_30.loc[start:end, coin].max() * 100, 1),
        })

breakaway_periods = pd.DataFrame(periods).sort_values("start").reset_index(drop=True)
breakaway_periods

To see one example clearly, I plot the coin with the most breakaway days: its price against BTC (both starting at 100) and its 30-day correlation, with the breakaway days shaded.

In [ ]:
top_coin = breakaway.sum().idxmax()
indexed = prices[[top_coin, "BTC"]] / prices[[top_coin, "BTC"]].iloc[0] * 100

fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
axes[0].plot(indexed.index, indexed[top_coin], label=top_coin)
axes[0].plot(indexed.index, indexed["BTC"], color="orange", label="BTC")
axes[0].set_title(top_coin + " vs BTC (both start at 100)")
axes[0].legend()
axes[1].plot(roll_30.index, roll_30[top_coin], color="black")
axes[1].axhline(0.4, color="red", linestyle="--", label="0.4 threshold")
axes[1].set_title(top_coin + ": 30-day correlation with BTC")
axes[1].legend()

# shade every breakaway day
for day in breakaway.index[breakaway[top_coin]]:
    for ax in axes:
        ax.axvspan(day, day + pd.Timedelta(days=1), color="green", alpha=0.2)
plt.show()

## 4.4 Hypothesis 7: breakaways come with unusual volume

If a coin breaks away because of real news or a new story, more people should be trading it. To compare coins with very different sizes, I divide each day's volume by the coin's **rolling 30-day median volume**. A value of 1.0 means a normal day, 2.0 means twice the usual volume.

Then I compare breakaway days with all other days. Volume data is very skewed (a few huge days), so instead of a t-test I use the **Mann-Whitney U test**. It compares the ranks of the values, so a few extreme days can't dominate it.

In [ ]:
# today's volume compared to the coin's usual volume
rel_volume = volumes[alts] / volumes[alts].rolling(30).median()
rel_volume = rel_volume.loc[breakaway.index]

# all coins in one long list: one row per coin per day
vol_long = pd.DataFrame({
    "rel_volume": rel_volume.stack(),
    "breakaway": breakaway.stack(),
}).dropna()

on_break = vol_long.loc[vol_long["breakaway"], "rel_volume"]
normal = vol_long.loc[~vol_long["breakaway"], "rel_volume"]
print("median relative volume on breakaway days:", round(on_break.median(), 2))
print("median relative volume on normal days:   ", round(normal.median(), 2))

u, p = stats.mannwhitneyu(on_break, normal, alternative="greater")
print("Mann-Whitney U p-value:", round(p, 6))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.boxplot(data=vol_long, x="breakaway", y="rel_volume", showfliers=False, ax=ax)
ax.set_xticks([0, 1], ["Normal days", "Breakaway days"])
ax.set_xlabel("")
ax.set_ylabel("volume / usual volume")
ax.set_title("Trading volume on breakaway vs normal days")
plt.show()

# the same comparison for each coin
per_coin = vol_long.groupby([vol_long.index.get_level_values("coin"), "breakaway"])["rel_volume"].median().unstack()
per_coin.columns = ["normal_days", "breakaway_days"]
per_coin.dropna().round(2)

## 4.5 Hypothesis 8: breakaways cluster by theme

Finally I group my coins by their **theme** (from `coin_categories.csv`) and compare: average correlation and beta, and how many breakaway days each theme had. If breakaways were random, every theme would get a similar share per coin.

In [ ]:
theme_table = beta_table.copy()
theme_table["breakaway_days"] = breakaway.sum()
theme_table["low_corr_days_%"] = (roll_30 < 0.4).mean() * 100

by_theme = theme_table.groupby("theme").agg(
    coins=("beta", "size"),
    avg_correlation=("correlation", "mean"),
    avg_beta=("beta", "mean"),
    breakaway_days=("breakaway_days", "sum"),
)
by_theme["breakaway_days_per_coin"] = by_theme["breakaway_days"] / by_theme["coins"]
by_theme = by_theme.sort_values("breakaway_days_per_coin", ascending=False).round(2)
by_theme

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(by_theme.index, by_theme["breakaway_days_per_coin"], color="seagreen")
ax.set_title("Breakaway days per coin, by theme")
ax.set_ylabel("days")
plt.xticks(rotation=20)
plt.show()

To check if the share of breakaway days really depends on the theme, I use a **chi-square test** on a table of theme × (breakaway day / normal day). A small p-value means the themes are not all the same.

In [ ]:
counts = vol_long.copy()
counts["theme"] = themes[counts.index.get_level_values("coin")].values
table = pd.crosstab(counts["theme"], counts["breakaway"])
print(table, "\n")

chi2, p, dof, expected = stats.chi2_contingency(table)
print("chi-square p-value:", p)

## 4.6 What I learned in Part B

**Hypothesis 4 (altcoins are highly correlated with BTC): mostly supported, with clear exceptions.** The average correlation of daily returns with BTC is 0.71, and all 11 links are statistically significant (every p-value is far below 0.05). The big coins follow BTC closely: ETH 0.91, SOL 0.87, XRP 0.86. But four coins sit much lower: TAO 0.55, ZEC 0.50, HYPE 0.47 and XMR 0.42.

**Hypothesis 5 (beta to BTC): most altcoins amplify BTC's moves.** 8 of my 11 altcoins have a beta above 1. ZEC is the wildest (1.59), then ADA (1.37). BNB (0.83) and XMR (0.79) move less than BTC. A high beta doesn't mean a coin follows BTC: ZEC has the highest beta but BTC explains only 25% of its daily moves (R²), compared to 82% for ETH.

**Hypothesis 6 (breakaways exist): supported.** With my rule (30-day correlation below 0.4 and beating BTC by more than 20% over 30 days) I found 16 breakaway periods and 107 breakaway days. Almost all of them belong to four coins: ZEC (33 days), HYPE (28), XMR (28) and TAO (17). BNB had a single day, and the other six coins had none. The strongest one was ZEC from 22 Sep to 6 Oct 2026, when it beat BTC by up to 98.9% over 30 days.

**Hypothesis 7 (breakaways come with unusual volume): weakly supported.** Relative volume on breakaway days had a median of 1.05 (vs 0.95 on normal days), and the Mann-Whitney test says the difference is real (p = 0.00003). But it's small: breakaways are not usually big volume explosions. TAO (3.2x) and BNB's single day (4.0x) did come with heavy volume, while HYPE's breakaways actually had lower volume than usual (0.80).

**Hypothesis 8 (breakaways cluster by theme): strongly supported.** Privacy coins had 30.5 breakaway days per coin, DEX/Perps (HYPE) 28 and AI (TAO) 17, while Layer 1, Meme, Oracle and Payments coins had zero. The chi-square test gives p ≈ 9e-40, so the share of breakaway days clearly depends on the theme.

**My takeaway:** the "everything follows Bitcoin" story is true for the big, established coins. The coins that break away are the ones with their own story (privacy, AI, perpetual futures trading). That fits what I saw in Part A: in 2026 there was no broad altseason, only a few theme coins going their own way.

**Limits:** I only have one year and 12 coins, my 0.4 / 20% thresholds are my own choice, and one coin per theme is a very small sample. Correlation also doesn't tell me *why* a coin broke away.

# 5. Insights and Recommendations

In this last section I put Part A and Part B together: a scorecard for all 8 hypotheses, two summary charts, my answer to why there was no full altseason, practical takeaways and the limits of my work.

## 5.1 Hypothesis scorecard

I rebuild the key numbers from the variables of Sections 3 and 4, so the table updates itself if the data changes.

In [ ]:
# Part A numbers
avg_years = complete.groupby("phase")["years"].mean()
alt_in_bull = (alt_periods["btc_phase"] == "Bull").sum()
late_in_bull = ((alt_periods["btc_phase"] == "Bull") & (alt_periods["position_in_phase_%"] >= 66)).sum()

# Part B numbers (recomputed, because p was reused in Section 4)
coins_breaking = (breakaway.sum() >= 5).sum()
vol_p = stats.mannwhitneyu(on_break, normal, alternative="greater").pvalue
chi_p = stats.chi2_contingency(table)[1]

scorecard = pd.DataFrame([
    ["H1: ~3 bull years, ~1 bear year", "drawdown rule (-50%, 180 days)",
     f"bull {avg_years['Bull']:.1f} y, bear {avg_years['Bear']:.1f} y",
     "supported" if abs(avg_years["Bull"] - 3) < 0.5 and abs(avg_years["Bear"] - 1) < 0.5 else "partly"],
    ["H2: bull and bear days differ", "Welch t-test + Levene",
     f"mean p = {p_value:.1e}, volatility p = {lev_p:.3f}",
     "partly" if p_value < 0.05 and lev_p > 0.01 else ("supported" if p_value < 0.05 else "not")],
    ["H3: altseasons come late in the bull", "ETH/BTC +50% in 90 days",
     f"{alt_in_bull} of {len(alt_periods)} in bulls, {late_in_bull} late",
     "supported" if late_in_bull > len(alt_periods) / 2 else ("partly" if alt_in_bull > 0 else "not")],
    ["H4: altcoins highly correlated with BTC", "Pearson r",
     f"average r = {corr_btc.mean():.2f}, lowest {corr_btc.idxmin()} {corr_btc.min():.2f}",
     "supported" if corr_btc.min() >= 0.6 else ("partly" if corr_btc.mean() >= 0.6 else "not")],
    ["H5: altcoins amplify BTC (beta > 1)", "beta = cov / var",
     f"{(beta_table['beta'] > 1).sum()} of {len(beta_table)} above 1",
     "supported" if (beta_table["beta"] > 1).mean() >= 0.8 else ("partly" if (beta_table["beta"] > 1).mean() >= 0.5 else "not")],
    ["H6: some coins break away", "rolling corr < 0.4 and +20% vs BTC",
     f"{len(breakaway_periods)} periods, {int(breakaway.sum().sum())} days, {coins_breaking} coins",
     "supported" if len(breakaway_periods) > 0 else "not"],
    ["H7: breakaways come with unusual volume", "Mann-Whitney U",
     f"median {on_break.median():.2f}x vs {normal.median():.2f}x, p = {vol_p:.1e}",
     "partly" if vol_p < 0.05 and on_break.median() / normal.median() < 1.5 else ("supported" if vol_p < 0.05 else "not")],
    ["H8: breakaways cluster by theme", "chi-square",
     f"p = {chi_p:.1e}, top theme {by_theme.index[0]}",
     "supported" if chi_p < 0.05 else "not"],
], columns=["hypothesis", "test used", "key number", "verdict"])

scorecard

In [ ]:
# quick count of the verdicts
scorecard["verdict"].value_counts()

## 5.2 Two charts that tie it all together

**Risk vs return.** For each of my 12 coins I annualize the last year of daily returns: volatility x √365 and average return x 365. Bubble size is the number of breakaway days, so the coins with their own story stand out.

In [ ]:
summary = pd.DataFrame({
    "ann_return_%": returns.mean() * 365 * 100,
    "ann_volatility_%": returns.std() * np.sqrt(365) * 100,
    "corr_btc": corr["BTC"],
    "beta": pd.Series(beta),
    "breakaway_days": breakaway.sum(),
    "theme": themes,
})
# BTC itself: beta 1, no breakaways
summary.loc["BTC", ["beta", "breakaway_days"]] = [1, 0]
summary = summary.round(2).sort_values("ann_return_%", ascending=False)
summary

In [ ]:
fig, ax = plt.subplots(figsize=(11, 7))
sns.scatterplot(data=summary, x="ann_volatility_%", y="ann_return_%", hue="theme",
                size="breakaway_days", sizes=(80, 900), alpha=0.7, ax=ax)

# write the coin name next to each bubble
for coin, row in summary.iterrows():
    ax.text(row["ann_volatility_%"] + 1, row["ann_return_%"], coin, fontsize=9)

ax.axhline(0, color="grey", linewidth=1)
ax.set_title("Risk vs return over the last year (bubble size = breakaway days)")
ax.set_xlabel("Annualized volatility (%)")
ax.set_ylabel("Annualized average return (%)")
ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8)
plt.show()

**Market map.** Correlation with BTC on one axis and beta on the other. Top right: coins that follow BTC and swing harder. Left: coins that live their own life.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 7))
alts_map = summary.drop("BTC")
sns.scatterplot(data=alts_map, x="corr_btc", y="beta", hue="theme", s=200, ax=ax)
for coin, row in alts_map.iterrows():
    ax.text(row["corr_btc"] + 0.01, row["beta"] + 0.02, coin, fontsize=9)

# my two dividing lines
ax.axvline(0.6, color="grey", linestyle="--")
ax.axhline(1, color="grey", linestyle="--")
ax.text(0.62, ax.get_ylim()[1] * 0.97, "follows BTC", fontsize=9)
ax.text(0.40, ax.get_ylim()[1] * 0.97, "own story", fontsize=9)

ax.set_title("Market map: how tightly each altcoin follows BTC and how hard it swings")
ax.set_xlabel("Correlation with BTC")
ax.set_ylabel("Beta to BTC")
ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8)
plt.show()

## 5.3 Why no full altseason in 2025-2026?

In [ ]:
above_line = (alt_score >= 75).sum()
print(f"average alt score: {alt_score.mean():.0f}%")
print(f"days at or above 75%: {above_line} of {len(alt_score)}")
print(f"ETH/BTC 90-day change today: {eth['change_90d'].iloc[-1]:.0%}")

# who did the breaking away
breakaway.sum().groupby(themes[alts]).sum().sort_values(ascending=False)

**What my data shows:**
- My altseason score (share of 11 altcoins beating BTC over 90 days) averaged about 45%, well below the usual 75% line, and stayed above it only for a short spike in Sep-Oct 2026.
- The breakaways came almost only from theme coins (privacy, AI, perps). The big, liquid altcoins (ETH, SOL, XRP, ADA, DOGE) stayed glued to BTC with correlations of 0.8-0.9.
- So money did flow into altcoins, but into a few stories at a time, not the whole market.

**Context, not tested in this notebook** (possible reasons from market news, my data cannot prove them):
- **Spot BTC ETFs** (US, since Jan 2024) pulled a lot of new money straight into BTC. That money often doesn't rotate into altcoins like retail money did in 2017 and 2021.
- **Too many new tokens.** Thousands of new tokens and meme coins split attention and money, so no single wave lifts all altcoins.
- **Theme rotation.** Money jumps from one narrative to the next (AI, privacy, perps DEXs), which looks exactly like my breakaway clusters: short, strong, and limited to a few coins.

## 5.4 Practical recommendations

*This is a data project, not financial advice.*

1. **Watch the 30-day rolling correlation.** When a coin's correlation with BTC drops below about 0.4 while it is beating BTC, something coin-specific is going on. That is my breakaway signal.
2. **Watch the altseason score, not only ETH/BTC.** A real altseason needs most altcoins beating BTC. A spike from a few coins is a rotation, not an altseason.
3. **Big altcoins are not diversifiers.** ETH, SOL and XRP behave like leveraged BTC. Holding them adds BTC risk, not protection.
4. **Theme coins can diversify, but they are risky.** Privacy, AI and perps coins had the lowest correlation with BTC, but also high volatility and short-lived runs.
5. **Size high-beta positions carefully.** A coin with beta 1.5 falls about 1.5% for each 1% BTC drop, so the same position size carries more risk.
6. **Respect the cycle.** Bull phases lasted about 3 years and bears about 1 year, and each bull return was smaller than the last. Volatility is about the same in both phases, so only the direction changes.

## 5.5 Limitations and next steps

**Limitations**
- Part B uses only one year of daily data for 12 coins (the free CoinGecko limit), and most themes have only one coin.
- Only 3 full BTC cycles exist, which is a very small sample for a "4-year cycle" claim.
- My thresholds (-50% for a bear, 0.4 correlation, +20% vs BTC, 75% alt score) are my own choices. Other values would give other dates.
- The alt score uses 11 coins, not the top 50 of the official index.
- Correlation is not causation: I did not test the ETF, token dilution or rotation reasons.

**Next steps**
- Use more coins and more years (an API key would unlock longer history).
- Test how sensitive the results are to the thresholds.
- Add news or social data to test the theme rotation idea.
- Turn the signals (rolling correlation, alt score) into a live Streamlit dashboard, which is the next part of this project.